In [3]:
##############
## INITIATE ##
##############

## Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch

## Import modules
from f_wsi_reader import slide_path_to_tiles_at_coordinates
from f_wsi_reader import get_tile_map
from f_wsi_reader import slide_to_tiles_at_coordinates
from f_wsi_reader import read_slide
from f_wsi_reader import get_dimensions
from f_wsi_reader import apply_all_filters
from f_wsi_reader import apply_all_transforms
from f_wsi_reader import reassemble_tiles
from f_wsi_reader import select_patches_on_grid

#
###

In [4]:
#####################################
## FUNCTIONS FOR MODEL APPLICATION ##
#####################################

## numpy_to_device
## Goal:
## Convert a list of numpy arrays to tensors suitable for model input on a specific device.
## Inputs:
## X (list of numpy arrays): List of images in numpy array format.
## Outputs:
## tensors (list of torch.Tensor): List of tensors formatted for model input.
def numpy_to_device(X):
    tensors = []
    for x in X:        
        x = cv2.resize(x, (1024, 1024))
        x = torch.tensor(x)        
        x = x.swapaxes(2, 1).swapaxes(1, 0)  # (H, W, C) --> (C, H, W)
        x = x.float()
        x = x[None, :, :, :]  # (C, H, W) --> (B=1, C, H, W)
        x = x.to(device)
        tensors += [x]        
    return tensors

## device_to_numpy
## Goal:
## Convert a list of tensors from a device back to numpy arrays.
## Inputs:
## X (list of torch.Tensor): List of tensors on a device.
## Outputs:
## images (list of numpy arrays): List of images converted to numpy format.
def device_to_numpy(X):
    images = []
    for x in X:
        x = x.cpu().numpy()
        x = x.swapaxes(0, 1).swapaxes(1, 2)    
        x = x.astype('uint8')
        x = cv2.resize(x, (tile_size_, tile_size_))
        images += [x]        
    return images

#
###

In [8]:
#####################################
## FUNCTIONS FOR OBJECT EXTRACTION ##
#####################################

## Imports
# from f_wsi_reader import overlay_shifted_copies
import numpy as np
import cv2

## polish
## Goal:
## Apply erosion to the image to polish the objects.
## Inputs:
## img (numpy array): Image to apply erosion on.
## Outputs:
## img (numpy array): Eroded image.
def polish(img):
    ## Erosion
    kernel = np.ones((5, 5), np.uint8)  # Adjust the kernel size if needed
    img = cv2.erode(img, kernel, iterations=1)
    img = img.astype(np.uint8)

    # ## Diffusion 
    # for k in range(3):
    #     img = overlay_shifted_copies(img) > 0
    # img = img.astype(np.uint8)
    # img = img.reshape(img.shape[0], img.shape[1], 1)
    
    return img

## extract_object_pixels
## Goal:
## Extract the pixel coordinates of objects from a binary segmentation mask.
## Inputs:
## mask (numpy.ndarray): A 2D binary segmentation mask where objects have a value of 1 (or 255) and background is 0.
## Outputs:
## object_pixels (list): A list of numpy arrays, each containing the (row, column) coordinates of the pixels of each object.
def extract_object_pixels(mask):
    ## Ensure the mask is binary (values are either 0 or 1)
    if mask.max() > 1:
        mask = mask // 255  # Convert 255 values to 1 for consistency if it's an 8-bit mask.

    ## Find connected components (objects) in the mask
    num_labels, labels = cv2.connectedComponents(mask.astype(np.uint8))

    ## Extract pixel coordinates for each object, ignoring the background (label 0)
    object_pixels = []
    for label in range(1, num_labels):
        ## Find pixel coordinates where the label is present
        pixels = np.column_stack(np.where(labels == label))
        object_pixels.append(pixels)

    return object_pixels

## get_bounding_box
## Goal:
## Get the bounding box of an object from its pixel coordinates.
## Inputs:
## coords (numpy.ndarray): Array of (row, column) coordinates of the object's pixels.
## Outputs:
## bounding_box (tuple): Bounding box coordinates (x_min, y_min, x_max, y_max).
def get_bounding_box(coords):
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    return x_min, y_min, x_max, y_max

## get_bounding_box_square
## Goal:
## Get the bounding box of an object from its pixel coordinates, making it square.
## Inputs:
## coords (numpy.ndarray): Array of (row, column) coordinates of the object's pixels.
## Outputs:
## bounding_box (tuple): Bounding box coordinates (x_min, y_min, x_max, y_max).
def get_bounding_box_square(coords):
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    
    ## Calculate width and height
    width = x_max - x_min
    height = y_max - y_min
    
    ## Make the bounding box square by taking the larger of the two dimensions
    side_length = max(width, height)
    
    ## Center the square box around the object's bounding box
    x_center = (x_min + x_max) // 2
    y_center = (y_min + y_max) // 2
    
    x_min = x_center - side_length // 2
    x_max = x_center + side_length // 2
    y_min = y_center - side_length // 2
    y_max = y_center + side_length // 2

    return x_min, y_min, x_max, y_max

## inlay_objects
## Goal:
## Visualise individual objects in an image.
## Inputs:
## img (numpy array): Image as a numpy array.
## coordinates (list): List of coordinates of pixels of objects.
## Outputs:
## img (numpy array): Image with objects inlaid in different colors.
def inlay_objects(img, coordinates):
    img = img * 0
    if len(img.shape) < 3:
        img = img[:, :, None]
    if img.shape[2] == 1: 
        img = np.repeat(img, 3, axis=2)
    for i in range(len(coordinates)):
        coordinates_i = coordinates[i].astype("int")
        color_vect = np.random.uniform(0, 1, 3) * 255
        img[coordinates_i[:, 0], coordinates_i[:, 1], 0] = color_vect[0]
        img[coordinates_i[:, 0], coordinates_i[:, 1], 1] = color_vect[1]
        img[coordinates_i[:, 0], coordinates_i[:, 1], 2] = color_vect[2]
    return img

#
###

In [9]:
###########
## PATHS ##
###########

## Input folders
wsi_paths = [
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,24856,26659,4475,4521)/reference/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,24856,26659,4475,4521).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,59171,63171,7533,6109)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,59171,63171,7533,6109).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,67585,64417,4452,4506)/reference/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,67585,64417,4452,4506).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,46495,20054,8837,6413)/reference/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,46495,20054,8837,6413).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,57282,31023,5529,3427)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,57282,31023,5529,3427).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,36959,41299,4761,4182)/reference/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,36959,41299,4761,4182).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/EU_18387_14_1E_HandE_TissueTrain_(1.00,37833,31517,4557,4584)/reference/EU_18387_14_1E_HandE_TissueTrain_(1.00,37833,31517,4557,4584).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/EU_26162_16_9x_HandE+-+2017-11-28+11.03.04_TissueTrain_(1.00,100353,23718,3132,2592)/reference/EU_26162_16_9x_HandE+-+2017-11-28+11.03.04_TissueTrain_(1.00,100353,23718,3132,2592).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/EU_29542_16_1S_HandE+-+2017-11-28+12.49.46_TissueTrain_(1.00,19271,22461,4395,4517)/reference/EU_29542_16_1S_HandE+-+2017-11-28+12.49.46_TissueTrain_(1.00,19271,22461,4395,4517).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/EU_7189_16_2D_HandE_TissueTrain_(1.00,49850,30011,6780,4562)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,49850,30011,6780,4562).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,30802,27560,4838,4043)/reference/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,30802,27560,4838,4043).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,51574,32631,3275,4229)/reference/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,51574,32631,3275,4229).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,55114,25216,4536,4338)/reference/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,55114,25216,4536,4338).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,56643,38213,4856,4743)/reference/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,56643,38213,4856,4743).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,65032,32714,5249,4496)/reference/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,65032,32714,5249,4496).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,79459,26002,6991,5312)/reference/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,79459,26002,6991,5312).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,96832,18250,5516,4626)/reference/17_A047-10719_16L+-+2017-05-11+08.56.52_TissueTrain_(1.00,96832,18250,5516,4626).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,22867,34300,4793,4231)/reference/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,22867,34300,4793,4231).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,31102,45887,5447,4555)/reference/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,31102,45887,5447,4555).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,32500,39761,6906,4907)/reference/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,32500,39761,6906,4907).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,70914,51067,4903,4392)/reference/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,70914,51067,4903,4392).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,72336,59698,4794,4144)/reference/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,72336,59698,4794,4144).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,34390,28093,5136,4145)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,34390,28093,5136,4145).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,35822,78102,5756,5827)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,35822,78102,5756,5827).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,36229,23829,6307,4503)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,36229,23829,6307,4503).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,38367,66246,6125,4896)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,38367,66246,6125,4896).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,38990,20011,4173,4043)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,38990,20011,4173,4043).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,60706,72991,5654,3861)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,60706,72991,5654,3861).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,60886,67539,4947,5608)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,60886,67539,4947,5608).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,76248,67823,10479,9468)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,76248,67823,10479,9468).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,80116,73851,6911,6321)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,80116,73851,6911,6321).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,113004,38996,5027,3923)/reference/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,113004,38996,5027,3923).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,20803,62435,5131,4209)/reference/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,20803,62435,5131,4209).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,26283,37137,5753,5015)/reference/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,26283,37137,5753,5015).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,32468,45045,4858,4233)/reference/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,32468,45045,4858,4233).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,66159,40946,5579,4805)/reference/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,66159,40946,5579,4805).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,135876,50767,7440,7332)/reference/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,135876,50767,7440,7332).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,22694,18333,1861,1842)/reference/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,22694,18333,1861,1842).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,30774,15012,3897,4556)/reference/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,30774,15012,3897,4556).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,35171,30052,4796,3922)/reference/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,35171,30052,4796,3922).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,55178,42697,4402,4501)/reference/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,55178,42697,4402,4501).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,90038,50737,7220,5954)/reference/17_A047-4463_153D+-+2017-05-11+09.40.22_TissueTrain_(1.00,90038,50737,7220,5954).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4519_1614P+-+2017-05-11+09.50.49_TissueTrain_(1.00,23106,22265,4869,4215)/reference/17_A047-4519_1614P+-+2017-05-11+09.50.49_TissueTrain_(1.00,23106,22265,4869,4215).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4519_1614P+-+2017-05-11+09.50.49_TissueTrain_(1.00,32465,16322,5478,4691)/reference/17_A047-4519_1614P+-+2017-05-11+09.50.49_TissueTrain_(1.00,32465,16322,5478,4691).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-4519_1614P+-+2017-05-11+09.50.49_TissueTrain_(1.00,40668,31510,4971,3976)/reference/17_A047-4519_1614P+-+2017-05-11+09.50.49_TissueTrain_(1.00,40668,31510,4971,3976).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,38749,9665,6205,3561)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,38749,9665,6205,3561).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,43956,9763,5196,4388)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,43956,9763,5196,4388).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,45204,11247,7140,1575)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,45204,11247,7140,1575).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,47148,7313,7312,5924)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,47148,7313,7312,5924).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,48496,35953,9058,5085)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,48496,35953,9058,5085).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,51407,8243,6784,4619)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,51407,8243,6784,4619).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,52577,14267,5085,3693)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,52577,14267,5085,3693).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,54519,9488,6257,5158)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,54519,9488,6257,5158).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,63070,11019,6762,4230)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,63070,11019,6762,4230).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,65666,35369,3712,2627)/reference/17_A047-47800_15J+-+2017-05-11+08.12.44_TissueTrain_(1.00,65666,35369,3712,2627).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,31536,28951,5007,5044)/reference/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,31536,28951,5007,5044).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,31805,24400,4623,5018)/reference/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,31805,24400,4623,5018).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,36982,20523,4778,4726)/reference/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,36982,20523,4778,4726).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,49303,45459,5801,4702)/reference/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,49303,45459,5801,4702).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,52930,58111,4962,4042)/reference/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,52930,58111,4962,4042).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,66137,72361,5359,4582)/reference/17_A047-8544_16J+-+2017-05-11+08.49.27_TissueTrain_(1.00,66137,72361,5359,4582).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_18387_14_1E_HandE_TissueTrain_(1.00,43743,25832,5070,4270)/reference/EU_18387_14_1E_HandE_TissueTrain_(1.00,43743,25832,5070,4270).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_18387_14_1E_HandE_TissueTrain_(1.00,66060,20075,6553,4555)/reference/EU_18387_14_1E_HandE_TissueTrain_(1.00,66060,20075,6553,4555).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_18387_14_1E_HandE_TissueTrain_(1.00,71293,13129,8510,6135)/reference/EU_18387_14_1E_HandE_TissueTrain_(1.00,71293,13129,8510,6135).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_18387_14_1E_HandE_TissueTrain_(1.00,76168,22923,5075,3762)/reference/EU_18387_14_1E_HandE_TissueTrain_(1.00,76168,22923,5075,3762).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_26162_16_9x_HandE+-+2017-11-28+11.03.04_TissueTrain_(1.00,27962,65821,5548,5102)/reference/EU_26162_16_9x_HandE+-+2017-11-28+11.03.04_TissueTrain_(1.00,27962,65821,5548,5102).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_26162_16_9x_HandE+-+2017-11-28+11.03.04_TissueTrain_(1.00,71575,40128,7478,9298)/reference/EU_26162_16_9x_HandE+-+2017-11-28+11.03.04_TissueTrain_(1.00,71575,40128,7478,9298).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_26162_16_9x_HandE+-+2017-11-28+11.03.04_TissueTrain_(1.00,79521,40158,7665,7682)/reference/EU_26162_16_9x_HandE+-+2017-11-28+11.03.04_TissueTrain_(1.00,79521,40158,7665,7682).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29373_16_1G_HandE+-+2017-11-28+10.25.19_TissueTrain_(1.00,76931,45185,4035,3028)/reference/EU_29373_16_1G_HandE+-+2017-11-28+10.25.19_TissueTrain_(1.00,76931,45185,4035,3028).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29373_16_1G_HandE+-+2017-11-28+10.25.19_TissueTrain_(1.00,83922,24972,3761,3246)/reference/EU_29373_16_1G_HandE+-+2017-11-28+10.25.19_TissueTrain_(1.00,83922,24972,3761,3246).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29373_16_1G_HandE+-+2017-11-28+10.25.19_TissueTrain_(1.00,87532,25547,2837,2718)/reference/EU_29373_16_1G_HandE+-+2017-11-28+10.25.19_TissueTrain_(1.00,87532,25547,2837,2718).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29542_16_1S_HandE+-+2017-11-28+12.49.46_TissueTrain_(1.00,75659,33158,4703,4374)/reference/EU_29542_16_1S_HandE+-+2017-11-28+12.49.46_TissueTrain_(1.00,75659,33158,4703,4374).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,38273,48519,5098,4066)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,38273,48519,5098,4066).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,38273,48519,8397,6610)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,38273,48519,8397,6610).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,38303,51678,4533,3746)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,38303,51678,4533,3746).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,42181,51411,4608,3983)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,42181,51411,4608,3983).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,42256,48498,4547,4087)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,42256,48498,4547,4087).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,50038,50739,6237,5062)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,50038,50739,6237,5062).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,55449,28387,4844,4528)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,55449,28387,4844,4528).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,85661,18988,2818,2779)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,85661,18988,2818,2779).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,89071,25109,3393,2713)/reference/EU_29769_15_1M_HandE+-+2017-11-28+11.56.21_TissueTrain_(1.00,89071,25109,3393,2713).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,16740,64053,7538,6445)/reference/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,16740,64053,7538,6445).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,49869,46001,7954,6351)/reference/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,49869,46001,7954,6351).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,60173,54175,7376,4343)/reference/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,60173,54175,7376,4343).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,67655,38820,10630,3966)/reference/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,67655,38820,10630,3966).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,85805,55583,8204,3965)/reference/EU_30479_16_G_HandE+-+2017-11-28+12.31.57_TissueTrain_(1.00,85805,55583,8204,3965).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,24941,64560,5738,5350)/reference/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,24941,64560,5738,5350).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,43070,65220,7175,6437)/reference/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,43070,65220,7175,6437).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,62504,54130,11631,10039)/reference/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,62504,54130,11631,10039).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,79440,62740,6856,7247)/reference/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,79440,62740,6856,7247).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,82097,43128,4845,4948)/reference/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,82097,43128,4845,4948).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,84457,50781,4247,7282)/reference/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,84457,50781,4247,7282).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,88969,62233,4857,6585)/reference/EU_35928_15_4N_HandE+-+2017-11-28+12.08.09_TissueTrain_(1.00,88969,62233,4857,6585).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_36244_16_9A_HandE+-+2017-11-28+12.41.47_TissueTrain_(1.00,49617,35825,6457,5141)/reference/EU_36244_16_9A_HandE+-+2017-11-28+12.41.47_TissueTrain_(1.00,49617,35825,6457,5141).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_36244_16_9A_HandE+-+2017-11-28+12.41.47_TissueTrain_(1.00,55018,52345,8888,4200)/reference/EU_36244_16_9A_HandE+-+2017-11-28+12.41.47_TissueTrain_(1.00,55018,52345,8888,4200).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_36244_16_9A_HandE+-+2017-11-28+12.41.47_TissueTrain_(1.00,59180,31894,4927,4484)/reference/EU_36244_16_9A_HandE+-+2017-11-28+12.41.47_TissueTrain_(1.00,59180,31894,4927,4484).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_36244_16_9A_HandE+-+2017-11-28+12.41.47_TissueTrain_(1.00,65097,34483,5342,2439)/reference/EU_36244_16_9A_HandE+-+2017-11-28+12.41.47_TissueTrain_(1.00,65097,34483,5342,2439).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,53950,17475,5146,3759)/reference/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,53950,17475,5146,3759).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,63062,11375,5880,10230)/reference/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,63062,11375,5880,10230).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,79336,6033,3293,2887)/reference/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,79336,6033,3293,2887).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,80608,10636,6031,6150)/reference/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,80608,10636,6031,6150).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,87859,4760,3773,3297)/reference/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,87859,4760,3773,3297).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,91820,3735,4015,2899)/reference/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,91820,3735,4015,2899).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,95823,4014,3210,2582)/reference/EU_37106_15_2K_HandE+-+2017-11-28+11.43.55_TissueTrain_(1.00,95823,4014,3210,2582).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37747_14_5S_HandE+-+2017-11-28+13.04.58_TissueTrain_(1.00,68252,45903,8230,5781)/reference/EU_37747_14_5S_HandE+-+2017-11-28+13.04.58_TissueTrain_(1.00,68252,45903,8230,5781).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37747_14_5S_HandE+-+2017-11-28+13.04.58_TissueTrain_(1.00,71629,58933,4041,3717)/reference/EU_37747_14_5S_HandE+-+2017-11-28+13.04.58_TissueTrain_(1.00,71629,58933,4041,3717).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37747_14_5S_HandE+-+2017-11-28+13.04.58_TissueTrain_(1.00,83034,43258,2911,3844)/reference/EU_37747_14_5S_HandE+-+2017-11-28+13.04.58_TissueTrain_(1.00,83034,43258,2911,3844).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_37747_14_5S_HandE+-+2017-11-28+13.04.58_TissueTrain_(1.00,91124,65380,6023,6520)/reference/EU_37747_14_5S_HandE+-+2017-11-28+13.04.58_TissueTrain_(1.00,91124,65380,6023,6520).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,27571,37697,5625,5326)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,27571,37697,5625,5326).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,34239,35703,5845,3734)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,34239,35703,5845,3734).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,38689,23348,4258,4405)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,38689,23348,4258,4405).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,38689,23348,7396,6807)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,38689,23348,7396,6807).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,38696,26631,4251,3555)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,38696,26631,4251,3555).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,41587,26599,4504,3461)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,41587,26599,4504,3461).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,41650,23361,4441,4377)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,41650,23361,4441,4377).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,59274,36066,4255,4001)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,59274,36066,4255,4001).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,59274,36066,8452,4001)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,59274,36066,8452,4001).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,63205,36233,4076,3785)/reference/EU_38663_17_7G_HandE+-+2017-11-28+11.48.55_TissueTrain_(1.00,63205,36233,4076,3785).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,116614,16450,2544,2254)/reference/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,116614,16450,2544,2254).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,119748,14269,3979,2096)/reference/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,119748,14269,3979,2096).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,35184,58159,4507,4194)/reference/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,35184,58159,4507,4194).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,47910,56732,4485,3363)/reference/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,47910,56732,4485,3363).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,51664,44297,8120,4036)/reference/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,51664,44297,8120,4036).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,51718,33744,8606,7364)/reference/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,51718,33744,8606,7364).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,51843,48791,5329,5031)/reference/EU_42619_15_2H_HandE+-+2017-11-28+11.31.12_TissueTrain_(1.00,51843,48791,5329,5031).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_52005_15_2I_HandE+-+2017-11-28+11.23.02_TissueTrain_(1.00,28397,19569,7429,4069)/reference/EU_52005_15_2I_HandE+-+2017-11-28+11.23.02_TissueTrain_(1.00,28397,19569,7429,4069).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_52005_15_2I_HandE+-+2017-11-28+11.23.02_TissueTrain_(1.00,82176,39556,3854,4967)/reference/EU_52005_15_2I_HandE+-+2017-11-28+11.23.02_TissueTrain_(1.00,82176,39556,3854,4967).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_52005_15_2I_HandE+-+2017-11-28+11.23.02_TissueTrain_(1.00,82176,39556,6644,4967)/reference/EU_52005_15_2I_HandE+-+2017-11-28+11.23.02_TissueTrain_(1.00,82176,39556,6644,4967).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_52005_15_2I_HandE+-+2017-11-28+11.23.02_TissueTrain_(1.00,86043,39559,3016,4926)/reference/EU_52005_15_2I_HandE+-+2017-11-28+11.23.02_TissueTrain_(1.00,86043,39559,3016,4926).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_7189_16_2D_HandE_TissueTrain_(1.00,42444,27793,4939,4562)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,42444,27793,4939,4562).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_7189_16_2D_HandE_TissueTrain_(1.00,42715,26030,13790,9645)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,42715,26030,13790,9645).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_7189_16_2D_HandE_TissueTrain_(1.00,44913,31266,5189,4018)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,44913,31266,5189,4018).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_7189_16_2D_HandE_TissueTrain_(1.00,45331,27124,4646,4687)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,45331,27124,4646,4687).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_7189_16_2D_HandE_TissueTrain_(1.00,49850,25743,6738,4269)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,49850,25743,6738,4269).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_7189_16_2D_HandE_TissueTrain_(1.00,56725,8687,6565,5284)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,56725,8687,6565,5284).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_7189_16_2D_HandE_TissueTrain_(1.00,57611,15937,4797,4449)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,57611,15937,4797,4449).png',
    '/Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/train/EU_7189_16_2D_HandE_TissueTrain_(1.00,63607,7966,7080,5364)/reference/EU_7189_16_2D_HandE_TissueTrain_(1.00,63607,7966,7080,5364).png',
]

## Output folder
output_folder = '/Users/willembonnaffe/Desktop/extracted_objects_5'

#
###

In [10]:
for wsi_path in wsi_paths:

    print('')
    print(f'Processing {wsi_path}')
    
    ################## 
    ## GET TILE MAP ##
    ################## 
    
    ## Open slide
    slide = read_slide(wsi_path)
    slide_size = get_dimensions(slide)
    print(slide_size)
    
    ## Parameters
    tile_size = 256
    
    ## Get tile_map
    thumbnail, tile_map, canvas_size = get_tile_map(slide, tile_size)
    
    ## Extract coordinates of pixels in tile_map that contain tissue
    mask = apply_all_filters(tile_map, low=10, std=15, high=200)
    mask = apply_all_transforms(mask)
    
    ## Get coordinates of tiles with tissue
    tile_coordinates = (np.argwhere(mask[:,:,0].T > 0) * tile_size).astype(int)
    tile_coordinates = [[tile_coordinate[0], tile_coordinate[1]] for tile_coordinate in tile_coordinates]
    
    ## Compute mask
    tile_map_masked = tile_map * mask
    thumbnail_masked = thumbnail * cv2.resize(mask, (thumbnail.shape[1], thumbnail.shape[0]), cv2.INTER_LINEAR)[:,:,None]
    tissue_mask = cv2.resize(mask, (slide_size[1], slide_size[0]), cv2.INTER_LINEAR)[:,:,None]    
    
    #
    ###
    
    ################################
    ## PATCH EXTRACTION FOR MODEL ##
    ################################
    
    ## Parameters
    level = 0
    ds = 1
    if level > 0: 
        ds = 4 * level
    tile_size_ = int(tile_size/ds)# + 64
    tile_size_ = 1024
    
    ## Remove patches too close to the edges
    tile_coordinates_ds = []
    for tile_coordinates_ in tile_coordinates:
        coords_0_lo = tile_coordinates_[0] + int(tile_size/2) - int(tile_size_/2)
        coords_0_hi = tile_coordinates_[0] + int(tile_size/2) + int(tile_size_/2)
        coords_1_lo = tile_coordinates_[1] + int(tile_size/2) - int(tile_size_/2)
        coords_1_hi = tile_coordinates_[1] + int(tile_size/2) + int(tile_size_/2)
        if (coords_0_lo > 0) & (coords_0_hi < canvas_size[1]) & (coords_1_lo > 0) & (coords_1_hi < canvas_size[0]):
            tile_coordinates_ds += [[coords_0_lo, coords_1_lo]]
    
    ## Get patches closest to grid points
    tile_coordinates_ds = select_patches_on_grid(tile_coordinates_ds, grid_spacing=512)
    print(f'Number of patches: {len(tile_coordinates_ds)}')
    
    ## Load patches from the dataset
    patches = slide_path_to_tiles_at_coordinates(wsi_path, tile_coordinates_ds, tile_size_, level) # [dataset[i] for i in range(min(num_patches_to_display, len(dataset)))]
       
    #
    ###
    
    ######################
    ## REASSEMBLE IMAGE ##
    ######################
    
    ## Reassemble image
    img = reassemble_tiles(patches, tile_coordinates_ds, canvas_size[1], canvas_size[0])
    
    #
    ###
    
    #################
    ## APPLY MODEL ##
    #################
        
    ## Models
    # from f_model import UNet as Model
    # pt_model_in = 'models/r10_3.pth'
    # num_classes = 4
    from f_model_transformer import UNetTransformer as Model
    pt_model_in = 'models/TransUNet_V2_0.pth'
    num_classes = 3
    
    ## Load model
    device = "mps"
    print(f"Using {device} device")
    model = Model(n_classes=num_classes).to(device)
    if os.path.exists(pt_model_in):
        model.load_state_dict(torch.load(pt_model_in))
        print("Loaded model")
    
    ## Example of iterating through the DataLoader
    preds = []
    k = 0
    X = numpy_to_device(patches)
    with torch.no_grad():
        for x in X:                
            Yhat = model(x)        
            Yhat = torch.sigmoid(Yhat) * 255 # Logits to probabilities to [0, 255]
            Yhat = device_to_numpy(Yhat)[0] # Only for single patch
            Yhat = np.argmax(Yhat, axis=2).reshape(Yhat.shape[0], Yhat.shape[1], 1) # Only for multiple classes
            preds += [Yhat]
    
    #
    ###
    
    ###########################
    ## VISUALISE PREDICTIONS ##
    ###########################
    
    ## Graphical parameters
    from matplotlib.colors import ListedColormap
    custom_cmap = 'gist_gray' # ListedColormap(['pink', 'hotpink', 'ivory'])
        
    #
    ###
    
    ############################
    ## REASSEMBLE PREDICTIONS ##
    ############################
    
    ## Assemble predictions
    mask = reassemble_tiles(preds, tile_coordinates_ds, canvas_size[1], canvas_size[0])
    mask = mask[:,:,0].reshape(mask.shape[0], mask.shape[1], 1) 
    mask[:tissue_mask.shape[0], :tissue_mask.shape[1]] = mask[:tissue_mask.shape[0], :tissue_mask.shape[1]] * tissue_mask
    
    #
    ###
      
    #################
    ## POLISH MASK ##
    #################
    
    ## Downscale factor
    ds = 4
    num_polish = 1
    
    ## Binarise mask
    mask_bin = ((mask > 0)*1).astype('uint8')
    mask_bin = cv2.resize(mask_bin, (int(mask_bin.shape[1]/ds), int(mask_bin.shape[0]/ds)))
    mask_bin = mask_bin[:,:,None]
    
    ## Polish mask
    for i in range(num_polish):
        mask_bin = polish(mask_bin)    
    
    #
    ###
    
    #####################
    ## EXTRACT OBJECTS ##
    #####################
    
    ## Get objects
    gland_coords = extract_object_pixels(mask_bin)
    print(len(gland_coords))
    
    ## Rescale coordinates to original size
    rescaled_gland_coords = [coords * ds for coords in gland_coords]
    
    #
    ###
    
    ##################
    ## SAVE OBJECTS ##
    ##################
    
    ## Create a folder to store extracted object images    
    os.makedirs(output_folder, exist_ok=True)
    
    ## Parameters
    buffer = 200
    dxy = 30
    
    ## Extract and save each object
    objects = []
    for i, coords in enumerate(rescaled_gland_coords):
    
        ## Discard small objects
        if len(coords) > 1000:
            
            ## Get bounding box of the object
            x_min, y_min, x_max, y_max = get_bounding_box_square(coords)
        
            ## Check overlap with boundary
            
            ## Discard object if overlaps tissue boundary
            if ((x_min - buffer) > 0) & ((x_max + buffer) < img.shape[1] + 1) & ((y_min - buffer) > 0) & ((y_max + buffer) < img.shape[0] + 1):
    
                ## Get object
                object_img = img[(y_min - dxy):(y_max + dxy), (x_min - dxy):(x_max + dxy)]
                object_img = cv2.cvtColor(object_img, cv2.COLOR_BGR2RGB) 
                objects += [object_img]
        
                ## Save the object image
                object_filename = os.path.join(output_folder, f'{os.path.basename(wsi_path)}_object_{i}.png')
                cv2.imwrite(object_filename, object_img)
    
                ## Update
                # print(f'Saved object {i} to {object_filename}')    
    
    #
    ###

print('')
print('Done')


Processing /Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,24856,26659,4475,4521)/reference/17_A047-19575_162E+-+2017-05-11+08.34.36_TissueTrain_(1.00,24856,26659,4475,4521).png
(4521, 4475)
Number of patches: 64
Using mps device
Loaded model
205

Processing /Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,59171,63171,7533,6109)/reference/17_A047-28956_156N+-+2017-05-11+09.57.53_TissueTrain_(1.00,59171,63171,7533,6109).png
(6109, 7533)
Number of patches: 154
Using mps device
Loaded model
291

Processing /Volumes/SED/BDI/databases/RAW/datasets/SEGG/images/test/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,67585,64417,4452,4506)/reference/17_A047-29957_152D+-+2017-05-11+10.06.25_TissueTrain_(1.00,67585,64417,4452,4506).png
(4506, 4452)
Number of patches: 64
Using mps device
Loaded model
162

Processing /Volumes/SED/BDI/databases/RAW/datasets